# Frame2Story: Multimodal Movie Recap Pipeline (Google Colab)

This notebook allows you to run the complete **Frame2Story** pipeline on Google Colab.

## ⚠️ TPU vs. GPU Runtime Recommendation
While this notebook is configured to run on Google Colab, please note the following regarding hardware accelerators:
- **TPU Runtime**: Since libraries like `faster-whisper` and `ultralytics` (YOLOv8) do not natively support TPU accelerators (which require TensorFlow/JAX/PyTorch-XLA integration), the pipeline will fall back and run on the **CPU** of the TPU VM. 
- **GPU Runtime (Recommended)**: We **highly recommend** choosing a **GPU Runtime (e.g., T4 GPU)** in Colab. The pipeline will automatically detect the GPU and run all inference tasks (Whisper audio transcription, YOLOv8 object detection, BART summarization, BERTScore evaluation, SentenceTransformers) on CUDA, making it **10x to 50x faster** without any code changes.

### How to change runtime type:
1. Go to the menu bar and select **Runtime** > **Change runtime type**.
2. Under **Hardware accelerator**, select **T4 GPU** (or TPU if you still prefer to run on CPU cores of the TPU VM).
3. Click **Save**.

## Step 1: Mount Google Drive
Mount your Google Drive to access the uploaded `Frame2Story` project folder.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Navigate to Project Directory
Change the directory to the path where you uploaded the `Frame2Story` folder in your Google Drive.

*Note: Adjust `PROJECT_PATH` if your folder name or structure is different.*

In [ ]:
import os

# Path to the Frame2Story project in your Google Drive
PROJECT_PATH = '/content/drive/MyDrive/Frame2Story'

if os.path.exists(PROJECT_PATH):
    os.chdir(PROJECT_PATH)
    print(f"Successfully changed directory to: {os.getcwd()}")
else:
    print(f"❌ ERROR: Path not found: {PROJECT_PATH}")
    print("Please make sure you have uploaded the project files and that the path is correct.")

## Step 3: Install Dependencies
Install the required Python libraries. 

*Note: We have pinned `scenedetect<0.6.0` in `requirements.txt` to prevent the `ImportError: cannot import name 'VideoManager'` caused by API removals in newer versions of PySceneDetect.*

In [ ]:
# Install project requirements
!pip install -r requirements.txt

# Ensure system-level FFmpeg is available (used for audio extraction & subtitles generation)
!apt-get update && apt-get install -y ffmpeg

## Step 4: Run the Pipeline
Now, you can execute the command-line pipeline. Make sure you have uploaded your target video to the `data/` folder (or edit the path below to point to your video).

### CLI Arguments:
- `--video`: Path to input video file (e.g., `data/Video Project.mp4`)
- `--subtitle`: Path to `.srt` subtitle file. Use `""` (empty string) if you want the pipeline to auto-generate subtitles from the audio using Whisper.
- `--progress`: Watch progress percentage (0-100)
- `--output_dir`: Output root directory for recap outputs

In [ ]:
# Adjust these paths and values as needed
VIDEO_PATH = "data/Video Project.mp4"
SUBTITLE_PATH = "data/generated_subtitles.srt"  # Change to "" if you want to auto-generate subtitles via Whisper
PROGRESS_PCT = "40"
OUTPUT_DIR = "outputs"

!python main_pipeline.py \
    --video "{VIDEO_PATH}" \
    --subtitle "{SUBTITLE_PATH}" \
    --progress {PROGRESS_PCT} \
    --output_dir "{OUTPUT_DIR}"

## Step 5: View Generated Recap
Once the pipeline completes execution, you can view the final generated recap text right here in the notebook.

In [ ]:
import os

recap_txt_path = os.path.join(OUTPUT_DIR, "final", "final_recap.txt")
recap_json_path = os.path.join(OUTPUT_DIR, "final", "final_recap.json")

if os.path.exists(recap_txt_path):
    print("=== FINAL RECAP ===\n")
    with open(recap_txt_path, 'r', encoding='utf-8') as f:
        print(f.read())
else:
    print(f"❌ Recap file not found at: {recap_txt_path}")
    print("Please check the pipeline output above for errors.")